# <center> Logistic Regression on simulated data
    
Authors: Rémi LELUC, François PORTIER

### Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from models import logisticReg
from sklearn.linear_model import LogisticRegression
from simus import run_log
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")
from tqdm.notebook import tqdm

### Simulated data for binary classification

In [2]:
# Parameters
n_samples = 5000   # number of samples
n_features = 20    # dimension of the problem
#n_features = 100    # dimension of the problem
# Simulate data for classification
seed=0
noise=0.01
X,y = make_classification(n_samples=n_samples,
                          n_features=n_features,
                          random_state=seed)
scaler = StandardScaler()
X=scaler.fit_transform(X)
y[y==0]=-1

## Compute optimal $\theta^\star$

In [3]:
𝜆 = 1/n_samples          #regularization parameter
c_reg = 1/(n_samples*λ)
log_sk = LogisticRegression(C=c_reg,fit_intercept=False,tol=1e-3)
# fit sklearn model
log_sk.fit(X=X,y=y)
coeff = log_sk.coef_[0]

In [4]:
data_term  = np.log(1+np.exp(np.multiply(-y,X@coeff))).mean()
reg_term = (𝜆/2)*sum(coeff**2)
print('data_term:',data_term)
print('reg_term :',reg_term)
# Optimal loss
log = logisticReg(X=X,y=y,λ=λ,fit_intercept=False)
loss_opt = log.loss(w=coeff)
print('loss_opt :',loss_opt)

data_term: 0.13860435751863479
reg_term : 0.0011374686749317507
loss_opt : 0.13974182619356654


## Parameter simulations

- n=5,000 and d=20:

N = int(2e2); N_exp = 5; lr = 1; k0 =20; alpha_power = 1; batch_size=32 ; batch_hess=32 ; c=5 ; power =1/2 ; 𝜂 = 1 ;same=False 

- n=5,000 and d=100:

N = int(2e2); N_exp = 5; lr = 1; k0 =25; alpha_power = 1; batch_size=32 ; batch_hess=64 
c=7 ; power =1/2 ; 𝜂 = 30 ;same=False 

In [5]:
N = int(2e2)    # number of passes over coordinates
N_exp = 30      # number of experiments
lr = 1          # numerator of learning rate
k0 =20          # smoothing parameter, denominator of learning rate
alpha_power = 1 # power in the learning rate
batch_size=32   # batch size for gradient estimates 
batch_hess=32   # batch size for hessian estimates
c=5             # numerator learning rate gamma_k for C_k
power =1/2      # power of learning rate gamma_k for C_k
𝜂 = 1           # importance weights parameter
same=False      # same batch for gradient/hessian

### SGD and Polyak-averaging

In [6]:
# SGD
w_sgd,loss_sgd = run_log(X=X,y=y,𝜆=𝜆,method='sgd',N_exp=N_exp,N=N,
                         batch_size=batch_size,batch_hess=batch_hess,lr=lr,c=c,k0=k0,
                         alpha_power=alpha_power,𝜂=None,power=None,fit_intercept=False,same=None)
# Polyak-averaging SGD
w_sgd_avg,loss_sgd_avg = run_log(X=X,y=y,𝜆=𝜆,method='sgd-avg',N_exp=N_exp,N=N,
                                 batch_size=batch_size,batch_hess=batch_hess,lr=lr,c=c,k0=k0,
                                 alpha_power=alpha_power,𝜂=None,power=None,fit_intercept=False,same=None)

### Conditioned-SGD with equal and adaptive weights

In [ ]:
# C-SGD equal
w_equal,loss_equal = run_log(X=X,y=y,𝜆=𝜆,method='csgd',N_exp=N_exp,N=N,
                             batch_size=batch_size,batch_hess=batch_hess,lr=lr,c=c,k0=k0,
                             alpha_power=alpha_power,𝜂=0,power=power,fit_intercept=False,same=same)

# C-SGD weighted
w_weighted,loss_weighted = run_log(X=X,y=y,𝜆=𝜆,method='csgd',N_exp=N_exp,N=N,
                                   batch_size=batch_size,batch_hess=batch_hess,lr=lr,c=c,k0=k0,
                                   alpha_power=alpha_power,𝜂=𝜂,power=power,fit_intercept=False,same=same)

### Save all results

In [ ]:
#np.save('results/logistic_sgd_n{}_d{}'.format(n_samples,n_features),loss_sgd-loss_opt)
#np.save('results/logistic_sgd_avg_n{}_d{}'.format(n_samples,n_features),loss_sgd_avg-loss_opt)
#np.save('results/logistic_equal_n{}_d{}'.format(n_samples,n_features),loss_equal-loss_opt)
#np.save('results/logistic_weighted_n{}_d{}'.format(n_samples,n_features),loss_weighted-loss_opt)

### Compute means and standard deviations over runs

In [ ]:
mean_sgd = np.mean(loss_sgd,axis=0)
mean_sgd_avg = np.mean(loss_sgd_avg,axis=0)
mean_equal = np.mean(loss_equal,axis=0)
mean_weighted = np.mean(loss_weighted,axis=0)

std_sgd = np.std(loss_sgd,axis=0)/2
std_sgd_avg = np.std(loss_sgd_avg,axis=0)/2
std_equal = np.std(loss_equal,axis=0)/2
std_weighted = np.std(loss_weighted,axis=0)/2

### Plot figures to check

In [ ]:
tab = np.arange(N+1)
fig,ax = plt.subplots(figsize=(4,4))
# SGD curves
plt.plot((mean_sgd-loss_opt)/(mean_sgd[0]-loss_opt),
         color='b',label='sgd',
         linestyle='-',marker='s',markevery=0.1,ms=4)
plt.fill_between(tab,
                 (mean_sgd-loss_opt-std_sgd)/(mean_sgd[0]-loss_opt),
                 (mean_sgd-loss_opt+std_sgd)/(mean_sgd[0]-loss_opt),
                  color='b',alpha=0.1)
# Polyak-averaging curves
plt.plot((mean_sgd_avg-loss_opt)/(mean_sgd_avg[0]-loss_opt),
         color='darkorange',label='sgd_avg',
         linestyle='--',marker='s',markevery=0.1,ms=4)
plt.fill_between(tab,
                 (mean_sgd_avg-loss_opt-std_sgd_avg)/(mean_sgd_avg[0]-loss_opt),
                 (mean_sgd_avg-loss_opt+std_sgd_avg)/(mean_sgd_avg[0]-loss_opt),
                  color='darkorange',alpha=0.1)
# C-SGD equal curves
plt.plot((mean_equal-loss_opt)/(mean_equal[0]-loss_opt),
         color='red',label=r'csgd($\eta$=0)',
         linestyle='-',marker='o',markevery=0.1,ms=4)
plt.fill_between(tab,
                 (mean_equal-loss_opt-std_equal)/(mean_equal[0]-loss_opt),
                 (mean_equal-loss_opt+std_equal)/(mean_equal[0]-loss_opt),
                  color='red',alpha=0.1)
# C-SGD weighted curves
plt.plot((mean_weighted-loss_opt)/(mean_weighted[0]-loss_opt),
         color='green',label=r'csgd($\eta$>0)',
         linestyle='-',marker='s',markevery=0.1,ms=4)
plt.fill_between(tab,
                 (mean_weighted-loss_opt-std_weighted)/(mean_weighted[0]-loss_opt),
                 (mean_weighted-loss_opt+std_weighted)/(mean_weighted[0]-loss_opt),
                  color='green',alpha=0.1)
# Graphics and Layout
plt.yscale('log')
plt.ylabel(r'Optimaliy Ratio',fontsize=15)
plt.xlabel('Iterations',fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.legend(fontsize=15)
plt.legend(loc='lower left',fontsize=10)
plt.grid(linestyle='--',which='both',alpha=0.5)
plt.tight_layout()
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

# Uncomment to save figure
#plt.savefig('logistic_n5000_d20.pdf',bbox_inches='tight',transparent=True,pad_inches=0)
#plt.savefig('logistic_n5000_d100.pdf',bbox_inches='tight',transparent=True,pad_inches=0)
plt.show()